# Feature Engineering EDA

Exploratory analysis of the tables produced by `FeatureENgineering.ipynb`.

**Prerequisite:** Run `FeatureENgineering.ipynb` first.

Covers:
1. Table inventory and row counts
2. Activity ontology tag distributions (journey signal, effort, persona hint, modality)
3. Journey state distributions across 30 / 90 / 180-day windows
4. Developer persona breakdown and confidence
5. Dormancy status and activation analysis
6. State transition patterns
7. Key feature distributions

## Section 0 — Setup

In [ ]:
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

con = duckdb.connect("developer_project.duckdb")
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:.3f}".format)

EXPECTED_TABLES = [
    "activity_enriched_v1", "activity_ontology_v1", "developer_universe_v1",
    "dev_features_30d_v1", "dev_features_90d_v1", "dev_features_180d_v1",
    "dev_profile_30d_v1", "dev_profile_90d_v1", "dev_profile_180d_v1",
    "dev_features_lifetime_v1", "dev_persona_v1",
    "dev_transition_v1", "dev_profile_final_v2",
    "dev_period_30d_features_v1", "dev_period_30d_profile_v1",
    "dev_period_30d_transitions_v1", "dev_weekly_features_v1",
    "dev_meaningful_week_v1", "dev_activation_v1",
    "dev_dormancy_base_v1", "dev_dormancy_status_v1",
    "dev_profile_final_v3"
]

available = set(con.execute("SHOW TABLES").fetchdf().iloc[:, 0].astype(str))
missing = [t for t in EXPECTED_TABLES if t not in available]
present = [t for t in EXPECTED_TABLES if t in available]
feature_like_tables = sorted(
    t for t in available
    if t.startswith("dev_") or t in {"activity_enriched_v1", "activity_ontology_v1", "developer_universe_v1"}
)

def table_columns(table_name):
    return con.execute(f"DESCRIBE {table_name}").df()["column_name"].astype(str).tolist()

def has_table(table_name):
    return table_name in available

def pick_first_column(table_name, candidates):
    cols = set(table_columns(table_name))
    for c in candidates:
        if c in cols:
            return c
    return None

def pick_existing_columns(table_name, candidates):
    cols = set(table_columns(table_name))
    return [c for c in candidates if c in cols]

final_t = "dev_profile_final_v3" if has_table("dev_profile_final_v3") else "dev_profile_final_v2" if has_table("dev_profile_final_v2") else None

print(f"Tables present : {len(present)} / {len(EXPECTED_TABLES)}")
if missing:
    print(f"Tables missing : {missing}")
else:
    print("All expected tables found.")
print(f"Feature-like tables available: {len(feature_like_tables)}")

---
## Section 1 — Table Inventory

In [ ]:
# Row counts for every feature engineering output table
rows = []
for t in present:
    n = con.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
    rows.append({"table": t, "row_count": n})

inventory = pd.DataFrame(rows).set_index("table")
print("Feature table row counts:")
display(inventory)

In [ ]:
# Developer universe coverage
universe = con.execute("SELECT COUNT(*) FROM developer_universe_v1").fetchone()[0]
contact  = con.execute("SELECT COUNT(*) FROM contact_final").fetchone()[0]
print(f"Developer universe (union of contact + activity): {universe:,}")
print(f"contact_final:                                    {contact:,}")
print(f"Coverage:                                         {universe/contact*100:.1f}%")

---
## Section 2 — Activity Ontology Tags

`activity_ontology_v1` assigns four behavioral tags to every activity event.
These tags drive journey state assignment and persona scoring.

In [ ]:
# Distribution of each ontology tag
tags = ["journey_signal", "effort_level", "persona_hint", "modality"]

for tag in tags:
    df = con.execute(f"""
        SELECT {tag}, COUNT(*) AS rows,
               ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct
        FROM activity_ontology_v1
        GROUP BY {tag}
        ORDER BY rows DESC
    """).df()
    print(f"\n{tag}")
    display(df)

In [ ]:
# Bar charts for all four tags
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

for ax, tag in zip(axes.flatten(), tags):
    df = con.execute(f"""
        SELECT {tag} AS label, COUNT(*) AS rows
        FROM activity_ontology_v1
        GROUP BY {tag} ORDER BY rows DESC
    """).df()
    ax.barh(df["label"].astype(str), df["rows"])
    ax.set_title(tag)
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1e6:.1f}M"))
    ax.invert_yaxis()

plt.suptitle("Activity Ontology Tag Distributions", fontsize=14)
plt.tight_layout()
plt.show()

---
## Section 3 — Journey State Distributions

Journey states are assigned per developer for three cumulative windows (30 / 90 / 180 days).
States: **Champion, Build, Evaluate, Learn, Discover, Dormant**

In [ ]:
# Journey state breakdown for each window
for d in [30, 90, 180]:
    table = f"dev_profile_{d}d_v1"
    if table not in present:
        print(f"{table} not found, skipping")
        continue
    df = con.execute(f"""
        SELECT journey_state,
               COUNT(*) AS developers,
               ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct
        FROM {table}
        GROUP BY journey_state
        ORDER BY developers DESC
    """).df()
    print(f"\n{d}-day window")
    display(df)

In [ ]:
# Side-by-side comparison of journey states across windows
all_states = set()
window_data = {}
for d in [30, 90, 180]:
    table = f"dev_profile_{d}d_v1"
    if table not in present:
        continue
    df = con.execute(f"""
        SELECT journey_state, COUNT(*) * 100.0 / SUM(COUNT(*)) OVER () AS pct
        FROM {table} GROUP BY journey_state
    """).df().set_index("journey_state")["pct"]
    window_data[f"{d}d"] = df
    all_states.update(df.index)

if window_data:
    comp = pd.DataFrame(window_data, index=sorted(all_states)).fillna(0)
    comp.plot(kind="bar", figsize=(12, 5))
    plt.title("Journey State % by Window")
    plt.ylabel("% of developers")
    plt.xticks(rotation=30)
    plt.legend(title="Window")
    plt.tight_layout()
    plt.show()

---
## Section 4 — Developer Personas

Personas are assigned from lifetime activity using weighted keyword scoring across six lanes:
**CUDA, GenAI, Robotics, Simulation, Learning/Community, Other**

In [ ]:
if "dev_persona_v1" in present:
    df = con.execute("""
        SELECT persona,
               COUNT(*) AS developers,
               ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct
        FROM dev_persona_v1
        GROUP BY persona
        ORDER BY developers DESC
    """).df()
    print("Persona distribution:")
    display(df)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].bar(df["persona"], df["developers"])
    axes[0].set_title("Developer Count by Persona")
    axes[0].tick_params(axis="x", rotation=30)
    axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1e6:.1f}M"))

    axes[1].pie(df["developers"], labels=df["persona"], autopct="%1.1f%%", startangle=140)
    axes[1].set_title("Persona Share")

    plt.tight_layout()
    plt.show()

In [ ]:
if "dev_persona_v1" in present:
    # Confidence tier breakdown
    cols = con.execute("DESCRIBE dev_persona_v1").df()["column_name"].tolist()

    if "persona_confidence_tier" in cols:
        conf = con.execute("""
            SELECT persona_confidence_tier,
                   COUNT(*) AS developers,
                   ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct
            FROM dev_persona_v1
            GROUP BY persona_confidence_tier
            ORDER BY developers DESC
        """).df()
        print("Persona confidence tiers:")
        display(conf)

    if "mixed_persona_flag" in cols:
        mixed = con.execute("""
            SELECT mixed_persona_flag, COUNT(*) AS developers
            FROM dev_persona_v1 GROUP BY mixed_persona_flag
        """).df()
        print("\nMixed persona flag:")
        display(mixed)

---
## Section 5 — Dormancy & Activation Analysis

Dormancy uses a survival-based framework with two thresholds:
- **Active**: last meaningful week < 56 days ago
- **At-Risk**: 56–83 days
- **Dormant**: ≥ 84 days
- **Unactivated**: never passed the activation gate

In [ ]:
if "dev_activation_v1" in present:
    act = con.execute("""
        SELECT is_activated,
               COUNT(*) AS developers,
               ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct
        FROM dev_activation_v1
        GROUP BY is_activated ORDER BY is_activated
    """).df()
    print("Activation status:")
    display(act)

    if "activation_reason" in con.execute("DESCRIBE dev_activation_v1").df()["column_name"].tolist():
        reason = con.execute("""
            SELECT activation_reason, COUNT(*) AS developers
            FROM dev_activation_v1
            GROUP BY activation_reason ORDER BY developers DESC
        """).df()
        print("\nActivation reason breakdown:")
        display(reason)

In [ ]:
if "dev_dormancy_status_v1" in present:
    dorm = con.execute("""
        SELECT dormancy_status,
               COUNT(*) AS developers,
               ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct
        FROM dev_dormancy_status_v1
        GROUP BY dormancy_status ORDER BY developers DESC
    """).df()
    print("Dormancy status breakdown:")
    display(dorm)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].bar(dorm["dormancy_status"], dorm["developers"])
    axes[0].set_title("Developers by Dormancy Status")
    axes[0].tick_params(axis="x", rotation=20)
    axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1e6:.1f}M"))

    axes[1].pie(dorm["developers"], labels=dorm["dormancy_status"], autopct="%1.1f%%", startangle=140)
    axes[1].set_title("Dormancy Share")

    plt.tight_layout()
    plt.show()

In [ ]:
# Distribution of days_since_last_meaningful_week for activated developers
if "dev_dormancy_base_v1" in present:
    days_df = con.execute("""
        SELECT days_since_last_meaningful_week
        FROM dev_dormancy_base_v1
        WHERE days_since_last_meaningful_week IS NOT NULL
          AND days_since_last_meaningful_week <= 365
    """).df()

    plt.figure(figsize=(12, 4))
    plt.hist(days_df["days_since_last_meaningful_week"], bins=60, edgecolor="none")
    plt.axvline(56, color="orange", linestyle="--", label="At-risk threshold (56d)")
    plt.axvline(84, color="red",    linestyle="--", label="Dormant threshold (84d)")
    plt.xlabel("Days since last meaningful active week")
    plt.ylabel("Developers")
    plt.title("Days Since Last Meaningful Activity (activated developers, ≤365d shown)")
    plt.legend()
    plt.tight_layout()
    plt.show()

---
## Section 6 — State Transition Patterns

In [ ]:
# Cumulative window transitions: 30d → 90d → 180d
if "dev_transition_v1" in present:
    cols = con.execute("DESCRIBE dev_transition_v1").df()["column_name"].tolist()
    print("dev_transition_v1 columns:", cols)

    if "state_30d" in cols and "state_90d" in cols:
        t30_90 = con.execute("""
            SELECT state_30d, state_90d, COUNT(*) AS developers
            FROM dev_transition_v1
            GROUP BY state_30d, state_90d
            ORDER BY developers DESC
            LIMIT 20
        """).df()
        print("\nTop transitions: 30d → 90d")
        display(t30_90)

    if "state_90d" in cols and "state_180d" in cols:
        t90_180 = con.execute("""
            SELECT state_90d, state_180d, COUNT(*) AS developers
            FROM dev_transition_v1
            GROUP BY state_90d, state_180d
            ORDER BY developers DESC
            LIMIT 20
        """).df()
        print("\nTop transitions: 90d → 180d")
        display(t90_180)

In [ ]:
# Period-to-period (non-cumulative 30-day buckets) transitions
if "dev_period_30d_transitions_v1" in present:
    transition_cols = table_columns("dev_period_30d_transitions_v1")
    from_col = pick_first_column("dev_period_30d_transitions_v1", ["from_state", "prev_journey_state"])
    to_col = pick_first_column("dev_period_30d_transitions_v1", ["to_state", "journey_state"])

    if from_col and to_col:
        period_t = con.execute(f"""
            SELECT {from_col} AS from_state,
                   {to_col} AS to_state,
                   COUNT(*) AS transitions,
                   ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct
            FROM dev_period_30d_transitions_v1
            WHERE {from_col} IS NOT NULL AND {to_col} IS NOT NULL
            GROUP BY 1, 2
            ORDER BY transitions DESC
            LIMIT 25
        """).df()
        print("Top period-to-period transitions:")
        display(period_t)
    elif "period_transition_type" in transition_cols:
        period_t = con.execute("""
            SELECT period_transition_type, COUNT(*) AS transitions,
                   ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct
            FROM dev_period_30d_transitions_v1
            GROUP BY period_transition_type
            ORDER BY transitions DESC
        """).df()
        print("Transition type distribution:")
        display(period_t)
    else:
        print("No recognizable transition columns found in dev_period_30d_transitions_v1")

---
## Section 7 — Key Feature Distributions

In [ ]:
# Lifetime feature summary statistics
if "dev_features_lifetime_v1" in present:
    summary = con.execute("""
        SELECT
            COUNT(*) AS developers,
            MIN(lifetime_activity_count)     AS min_activities,
            MAX(lifetime_activity_count)     AS max_activities,
            AVG(lifetime_activity_count)     AS avg_activities,
            PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY lifetime_activity_count) AS median_activities,
            MIN(lifetime_activity_score_sum) AS min_score,
            MAX(lifetime_activity_score_sum) AS max_score,
            AVG(lifetime_activity_score_sum) AS avg_score
        FROM dev_features_lifetime_v1
    """).df()
    print("Lifetime feature summary:")
    display(summary)

In [ ]:
# Average activity count per developer across time windows
window_stats = []
for d in [30, 90, 180]:
    table = f"dev_features_{d}d_v1"
    if table not in present:
        continue
    cols = con.execute(f"DESCRIBE {table}").df()["column_name"].tolist()
    if "activity_count_total" not in cols:
        continue
    row = con.execute(f"""
        SELECT
            '{d}d' AS window,
            COUNT(*) AS developers,
            AVG(activity_count_total) AS avg_activity_count,
            PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY activity_count_total) AS median_activity_count,
            SUM(CASE WHEN activity_count_total = 0 THEN 1 ELSE 0 END) AS zero_activity_developers
        FROM {table}
    """).fetchone()
    window_stats.append(row)

if window_stats:
    print("Activity count stats by window:")
    display(pd.DataFrame(window_stats, columns=["window", "developers", "avg_activity_count",
                                                 "median_activity_count", "zero_activity_developers"]))

In [ ]:
# Final profile summary
final_table = "dev_profile_final_v3" if "dev_profile_final_v3" in present else \
              "dev_profile_final_v2" if "dev_profile_final_v2" in present else None

if final_table:
    print(f"Final profile table: {final_table}")
    n = con.execute(f"SELECT COUNT(*) FROM {final_table}").fetchone()[0]
    print(f"Rows: {n:,}")
    print("\nColumns:")
    display(con.execute(f"DESCRIBE {final_table}").df()[["column_name", "column_type"]])
    print("\nSample rows:")
    display(con.execute(f"SELECT * FROM {final_table} LIMIT 5").df())

---
## Section 8 — Score Axis Decomposition

The PDF framework replaces a single `activity_score` with four independent axes so developers
with the same lifetime score can still be at very different journey stages:

| Axis | Signal |
|------|--------|
| Learn | `journey_signal` in Learn / Evaluate |
| Build | `journey_signal` = Build |
| Community | `journey_signal` = Champion |
| High-effort | `effort_level` = High |


In [ ]:
if "dev_features_lifetime_v1" in present:
    df = con.execute("""
        SELECT lifetime_learn_count, lifetime_evaluate_count,
               lifetime_build_count, lifetime_champion_count,
               lifetime_high_effort_count
        FROM dev_features_lifetime_v1
        WHERE lifetime_activity_count > 0
        USING SAMPLE 50000
    """).df()

    total_sig = (
        df["lifetime_learn_count"] + df["lifetime_evaluate_count"] +
        df["lifetime_build_count"] + df["lifetime_champion_count"]
    ).replace(0, float("nan"))
    df["learn_pct"]     = (df["lifetime_learn_count"] + df["lifetime_evaluate_count"]) / total_sig * 100
    df["build_pct"]     = df["lifetime_build_count"]   / total_sig * 100
    df["community_pct"] = df["lifetime_champion_count"] / total_sig * 100

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle("Score Axis Decomposition (active devs, 50k sample)", fontsize=13)

    means = {"Learn+Eval": df["learn_pct"].mean(),
             "Build": df["build_pct"].mean(),
             "Community": df["community_pct"].mean()}
    axes[0].bar(means.keys(), means.values(), color=["#42A5F5", "#66BB6A", "#EF5350"])
    axes[0].set_ylabel("Mean % of signalled activities")
    axes[0].set_title("Average Mix Across Active Developers")

    sample = df.dropna(subset=["learn_pct","build_pct","community_pct"]).sample(min(20000, len(df)))
    sc = axes[1].scatter(sample["learn_pct"], sample["build_pct"],
                         c=sample["community_pct"], cmap="RdYlGn", alpha=0.3, s=3)
    plt.colorbar(sc, ax=axes[1], label="Community %")
    axes[1].set_xlabel("Learn + Evaluate % of activities")
    axes[1].set_ylabel("Build % of activities")
    axes[1].set_title("Learn vs Build (coloured by Community)")

    hec = df["lifetime_high_effort_count"].clip(upper=50)
    axes[2].hist(hec, bins=30, color="#FFA726", edgecolor="white")
    axes[2].set_xlabel("High-effort activity count (capped at 50)")
    axes[2].set_ylabel("Developers")
    axes[2].set_title("High-effort Activity Distribution")

    plt.tight_layout()
    plt.show()
else:
    print("dev_features_lifetime_v1 not available.")


---
## Section 9 — Lane x Journey Stage Cross-Analysis

Combining persona lane with journey state shows where each technical community sits in its
adoption cycle and which lanes have the most developers stuck in early stages vs already
building or championing.


In [ ]:
if "dev_persona_v1" in present and final_t:
    state_col = pick_first_column(final_t, [
        "recent_journey_state_90d",
        "journey_state_90d",
        "current_journey_state_30d",
        "journey_state"
    ])

    if state_col:
        cross = con.execute(f"""
            SELECT p.persona,
                   f.{state_col} AS journey_state,
                   COUNT(*) AS n
            FROM dev_persona_v1 p
            JOIN {final_t} f USING (developer_id)
            WHERE p.persona IS NOT NULL AND f.{state_col} IS NOT NULL
            GROUP BY p.persona, f.{state_col}
        """).df()

        pivot = cross.pivot_table(index="persona", columns="journey_state", values="n", fill_value=0)
        pivot_pct = pivot.div(pivot.sum(axis=1), axis=0) * 100
        order = [c for c in ["Champion", "Build", "Evaluate", "Learn", "Discover", "Dormant"] if c in pivot_pct.columns]
        if order:
            pivot_pct = pivot_pct[order]
        colors = ["#AB47BC", "#EF5350", "#FFEE58", "#42A5F5", "#90A4AE", "#78909C"]

        ax = pivot_pct.plot(kind="bar", stacked=True, figsize=(12, 5),
                            color=colors[:len(pivot_pct.columns)], edgecolor="white", linewidth=0.4)
        ax.set_xlabel("Developer persona")
        ax.set_ylabel("% of developers")
        ax.set_title("Journey State by Lane")
        ax.legend(title="Journey state", bbox_to_anchor=(1.02,1), loc="upper left")
        plt.xticks(rotation=20, ha="right")
        plt.tight_layout()
        plt.show()
    else:
        print(f"No journey-state column found in {final_t}")
else:
    print("dev_persona_v1 or final profile table not available.")


---
## Section 10 — Activity Breadth and Cadence

**Breadth** measures cross-lane diversity. A developer touching CUDA, GenAI, and Simulation
is likely an enterprise platform team or broad researcher.

**Cadence** measures engagement regularity via the coefficient of variation (CV) of weekly
activity counts. Low CV = regular weekly engagement. High CV = occasional bursts.

In [ ]:
if "dev_features_lifetime_v1" in present:
    breadth = con.execute("""
        SELECT
            (CASE WHEN cuda_score > 0 THEN 1 ELSE 0 END
           + CASE WHEN genai_score > 0 THEN 1 ELSE 0 END
           + CASE WHEN robotics_score > 0 THEN 1 ELSE 0 END
           + CASE WHEN simulation_score > 0 THEN 1 ELSE 0 END
           + CASE WHEN learning_community_score > 0 THEN 1 ELSE 0 END) AS active_lanes,
            COUNT(*) AS n
        FROM dev_features_lifetime_v1
        WHERE lifetime_activity_count > 0
        GROUP BY active_lanes ORDER BY active_lanes
    """).df()
    total_b = breadth["n"].sum()

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle("Activity Breadth and Cadence", fontsize=13)

    axes[0].bar(breadth["active_lanes"].astype(str), breadth["n"] / total_b * 100,
               color="#42A5F5", edgecolor="white")
    axes[0].set_xlabel("Number of distinct technical lanes active")
    axes[0].set_ylabel("% of active developers")
    axes[0].set_title("Lane Breadth Distribution")

    if "dev_weekly_features_v1" in present:
        cadence = con.execute("""
            SELECT cv_bucket, COUNT(*) AS n FROM (
                SELECT developer_id,
                       ROUND(STDDEV(activity_count_total) / NULLIF(AVG(activity_count_total),0), 1) AS cv_bucket
                FROM dev_weekly_features_v1
                GROUP BY developer_id
                HAVING COUNT(*) >= 4
            ) WHERE cv_bucket IS NOT NULL AND cv_bucket <= 5
            GROUP BY cv_bucket ORDER BY cv_bucket
        """).df()
        if len(cadence):
            axes[1].bar(cadence["cv_bucket"].astype(str), cadence["n"],
                       color="#FFA726", edgecolor="white")
            axes[1].set_xlabel("Weekly activity CV (0=regular, >2=bursty)")
            axes[1].set_ylabel("Developers (>= 4 active weeks)")
            axes[1].set_title("Activity Cadence Distribution")
        else:
            axes[1].text(0.5, 0.5, "No data", ha="center", va="center")
            axes[1].axis("off")
    else:
        axes[1].text(0.5, 0.5, "dev_weekly_features_v1 not available", ha="center", va="center")
        axes[1].axis("off")

    plt.tight_layout()
    plt.show()

    multi = breadth[breadth["active_lanes"] > 1]["n"].sum() / total_b * 100
    print(f"Developers active in > 1 lane: {multi:.1f}%")
else:
    print("dev_features_lifetime_v1 not available.")

---
## Section 11 — Marketing Touch vs Self-Directed Activity

Some developers appear highly engaged only because they respond to campaigns. Using
`lead_source` from `activity_ontology_v1` to separate campaign-driven from organic activity.

In [ ]:
ao_cols = {r[0] for r in con.execute("DESCRIBE activity_ontology_v1").fetchall()}
if "lead_source" in ao_cols:
    intent = con.execute("""
        SELECT developer_id,
               COUNT(*) AS total_act,
               SUM(CASE WHEN lead_source NOT IN ('unknown','') AND lead_source IS NOT NULL
                        THEN 1 ELSE 0 END) AS campaign_act
        FROM activity_ontology_v1
        WHERE developer_id IS NOT NULL
        GROUP BY developer_id
    """).df()
    intent["campaign_pct"] = intent["campaign_act"] / intent["total_act"].clip(lower=1) * 100

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle("Marketing Touch vs Self-Directed Activity", fontsize=13)

    axes[0].hist(intent["campaign_pct"].clip(upper=100), bins=20,
                 color="#EF5350", edgecolor="white")
    axes[0].set_xlabel("Campaign-driven activity % per developer")
    axes[0].set_ylabel("Developers")
    axes[0].set_title("Campaign Activity Distribution")

    cp = intent["campaign_pct"]
    buckets = [("Fully organic (0%)",    (cp == 0).sum()),
               ("Mostly organic (<25%)", ((cp > 0) & (cp < 25)).sum()),
               ("Mixed (25-75%)",        ((cp >= 25) & (cp <= 75)).sum()),
               ("Mostly campaign (>75%)",(cp > 75).sum())]
    lbs, cnts = zip(*buckets)
    axes[1].barh(lbs, cnts, color=["#66BB6A", "#42A5F5", "#FFA726", "#EF5350"])
    axes[1].set_xlabel("Number of developers")
    axes[1].set_title("Intent Segmentation by Campaign Mix")

    plt.tight_layout()
    plt.show()
    print(f"Fully organic: {(cp==0).sum()/len(intent)*100:.1f}%")
else:
    print("lead_source column not in activity_ontology_v1 ? skipping.")

---
## Section 12 — Lane by Journey Stage

This section shows how each persona lane is distributed across journey stages.

In [ ]:
final_table = final_t
state_col = pick_first_column(final_table, ["recent_journey_state_90d", "current_journey_state_30d", "historical_journey_state_180d", "journey_state"]) if final_table else None
if final_table and state_col and {"persona", state_col}.issubset(set(table_columns(final_table))):
    lane_stage = con.execute(f"""
        SELECT persona, {state_col} AS journey_state, COUNT(*) AS developers
        FROM {final_table}
        WHERE persona IS NOT NULL AND {state_col} IS NOT NULL
        GROUP BY persona, {state_col}
    """).df()
    display(lane_stage.sort_values(["persona", "developers"], ascending=[True, False]))
    pivot = lane_stage.pivot_table(index="persona", columns="journey_state", values="developers", fill_value=0)
    pivot_pct = pivot.div(pivot.sum(axis=1), axis=0) * 100
    order = [c for c in ["Champion", "Build", "Evaluate", "Learn", "Discover", "Dormant"] if c in pivot_pct.columns]
    if order:
        pivot_pct = pivot_pct[order]
    pivot_pct.plot(kind="bar", stacked=True, figsize=(12, 5), edgecolor="white")
    plt.ylabel("% of developers within lane")
    plt.title("Journey stage composition by persona lane")
    plt.xticks(rotation=20, ha="right")
    plt.tight_layout()
    plt.show()
else:
    print("Could not find a final profile table with persona and journey-state columns.")


**What this section looks at:** It shows whether each technical lane is concentrated in Discover/Learn/Evaluate versus Build/Champion or Dormant states.

**Exact features / columns used:** `persona` plus the chosen journey-state column from the final profile table, usually `recent_journey_state_90d` and, if needed, `current_journey_state_30d` or `historical_journey_state_180d` from `dev_profile_final_v3` or `dev_profile_final_v2`.

**Why it matters:** This is one of the clearest business views of the feature system. It tells you which technical communities are early in the funnel, which are mature builders, and which are falling dormant.

---
## Section 13 — Journey Stage by Effort Level

This section shows how effort levels distribute across journey stages using the activity ontology table.

In [ ]:
if "activity_ontology_v1" in available:
    ao_cols = set(table_columns("activity_ontology_v1"))
    if {"journey_signal", "effort_level"}.issubset(ao_cols):
        stage_effort = con.execute("""
            SELECT journey_signal, effort_level, COUNT(*) AS rows
            FROM activity_ontology_v1
            WHERE journey_signal IS NOT NULL AND effort_level IS NOT NULL
            GROUP BY journey_signal, effort_level
        """).df()
        display(stage_effort.sort_values(["journey_signal", "rows"], ascending=[True, False]))
        pivot = stage_effort.pivot_table(index="journey_signal", columns="effort_level", values="rows", fill_value=0)
        pivot_pct = pivot.div(pivot.sum(axis=1), axis=0) * 100
        pivot_pct.plot(kind="bar", stacked=True, figsize=(11, 5), edgecolor="white")
        plt.ylabel("% of activities within journey stage")
        plt.title("Effort mix by journey stage")
        plt.xticks(rotation=20, ha="right")
        plt.tight_layout()
        plt.show()
    else:
        print("journey_signal and effort_level were not both found in activity_ontology_v1.")
else:
    print("activity_ontology_v1 not available.")


**What this section looks at:** It compares the low / moderate / high-effort mix across the journey stages.

**Exact features / columns used:** `journey_signal` and `effort_level` from `activity_ontology_v1`.

**Why it matters:** If the journey system is meaningful, later stages should generally show a heavier share of higher-effort engagement than early discovery-oriented stages.

---
## Section 14 — Lane by Modality

This section shows how each persona lane engages across modalities such as downloads, hosted APIs, training, and cloud workspace patterns.

In [ ]:
if "activity_ontology_v1" in available:
    ao_cols = set(table_columns("activity_ontology_v1"))
    if {"persona_hint", "modality"}.issubset(ao_cols):
        lane_modality = con.execute("""
            SELECT persona_hint, modality, COUNT(*) AS rows
            FROM activity_ontology_v1
            WHERE persona_hint IS NOT NULL AND modality IS NOT NULL
            GROUP BY persona_hint, modality
        """).df()
        display(lane_modality.sort_values(["persona_hint", "rows"], ascending=[True, False]))
        pivot = lane_modality.pivot_table(index="persona_hint", columns="modality", values="rows", fill_value=0)
        pivot_pct = pivot.div(pivot.sum(axis=1), axis=0) * 100
        pivot_pct.plot(kind="bar", stacked=True, figsize=(12, 5), edgecolor="white")
        plt.ylabel("% of activities within lane")
        plt.title("Modality mix by persona lane")
        plt.xticks(rotation=20, ha="right")
        plt.tight_layout()
        plt.show()
    else:
        print("persona_hint and modality were not both found in activity_ontology_v1.")
else:
    print("activity_ontology_v1 not available.")


**What this section looks at:** It shows whether each lane primarily engages through downloads, training, hosted APIs, cloud workspaces, or other modalities.

**Exact features / columns used:** `persona_hint` and `modality` from `activity_ontology_v1`.

**Why it matters:** This makes the lanes operationally interpretable. It tells you not just who developers are, but how they tend to interact with the ecosystem.

---
## Section 15 — Top Activities by Persona Lane

This section surfaces the most common activities inside each persona lane.

In [ ]:
if "activity_ontology_v1" in available:
    ao_cols = set(table_columns("activity_ontology_v1"))
    if {"persona_hint", "activity"}.issubset(ao_cols):
        top_lane_acts = con.execute("""
            WITH ranked AS (
                SELECT persona_hint, activity, COUNT(*) AS rows,
                       ROW_NUMBER() OVER (PARTITION BY persona_hint ORDER BY COUNT(*) DESC) AS rn
                FROM activity_ontology_v1
                WHERE persona_hint IS NOT NULL AND activity IS NOT NULL
                GROUP BY persona_hint, activity
            )
            SELECT * FROM ranked
            WHERE rn <= 8
            ORDER BY persona_hint, rows DESC
        """).df()
        display(top_lane_acts)
        personas = top_lane_acts["persona_hint"].drop_duplicates().tolist()[:6]
        fig, axes = plt.subplots(len(personas), 1, figsize=(10, 3.2 * len(personas)))
        if len(personas) == 1:
            axes = [axes]
        for ax, persona in zip(axes, personas):
            subset = top_lane_acts[top_lane_acts["persona_hint"] == persona].sort_values("rows")
            ax.barh(subset["activity"], subset["rows"], color="#42A5F5")
            ax.set_title(persona)
            ax.set_xlabel("Activity rows")
        plt.suptitle("Top activities by persona lane", fontsize=14)
        plt.tight_layout()
        plt.show()
    else:
        print("persona_hint and activity were not both found in activity_ontology_v1.")
else:
    print("activity_ontology_v1 not available.")


**What this section looks at:** It lists and plots the most common activity types within each lane.

**Exact features / columns used:** `persona_hint` and `activity` from `activity_ontology_v1`, ranked by row counts within each lane.

**Why it matters:** This is one of the best interpretability checks. It helps confirm whether a lane really looks like CUDA, GenAI, Robotics, Simulation, or Learning/Community in terms of observed behavior.

---
## Section 16 — Top Activities by Journey Stage

This section shows which activities are most associated with each journey stage.

In [ ]:
if "activity_ontology_v1" in available:
    ao_cols = set(table_columns("activity_ontology_v1"))
    if {"journey_signal", "activity"}.issubset(ao_cols):
        top_stage_acts = con.execute("""
            WITH ranked AS (
                SELECT journey_signal, activity, COUNT(*) AS rows,
                       ROW_NUMBER() OVER (PARTITION BY journey_signal ORDER BY COUNT(*) DESC) AS rn
                FROM activity_ontology_v1
                WHERE journey_signal IS NOT NULL AND activity IS NOT NULL
                GROUP BY journey_signal, activity
            )
            SELECT * FROM ranked
            WHERE rn <= 8
            ORDER BY journey_signal, rows DESC
        """).df()
        display(top_stage_acts)
        stages = top_stage_acts["journey_signal"].drop_duplicates().tolist()[:6]
        fig, axes = plt.subplots(len(stages), 1, figsize=(10, 3.2 * len(stages)))
        if len(stages) == 1:
            axes = [axes]
        for ax, stage in zip(axes, stages):
            subset = top_stage_acts[top_stage_acts["journey_signal"] == stage].sort_values("rows")
            ax.barh(subset["activity"], subset["rows"], color="#66BB6A")
            ax.set_title(stage)
            ax.set_xlabel("Activity rows")
        plt.suptitle("Top activities by journey stage", fontsize=14)
        plt.tight_layout()
        plt.show()
    else:
        print("journey_signal and activity were not both found in activity_ontology_v1.")
else:
    print("activity_ontology_v1 not available.")


**What this section looks at:** It shows the dominant activities that make up each journey stage.

**Exact features / columns used:** `journey_signal` and `activity` from `activity_ontology_v1`, ranked by row counts within each stage.

**Why it matters:** This validates the journey taxonomy. If the activities inside a stage do not match the intended meaning of Discover, Learn, Evaluate, Build, or Champion, the stage logic may need revision.

---
## Section 17 — Dormancy by Persona Heatmap

This section visualizes how dormancy risk concentrates across the persona lanes.

In [ ]:
if final_t and {"persona", "dormancy_status"}.issubset(set(table_columns(final_t))):
    lane_dorm = con.execute(f"""
        SELECT persona, dormancy_status, COUNT(*) AS developers
        FROM {final_t}
        WHERE persona IS NOT NULL AND dormancy_status IS NOT NULL
        GROUP BY persona, dormancy_status
    """).df()
    pivot = lane_dorm.pivot_table(index="persona", columns="dormancy_status", values="developers", fill_value=0)
    pivot_pct = pivot.div(pivot.sum(axis=1), axis=0) * 100
    plt.figure(figsize=(10, max(4, len(pivot_pct) * 0.8)))
    plt.imshow(pivot_pct, aspect="auto", cmap="YlOrRd", vmin=0, vmax=max(1, float(pivot_pct.to_numpy().max())))
    plt.xticks(range(len(pivot_pct.columns)), pivot_pct.columns, rotation=25, ha="right")
    plt.yticks(range(len(pivot_pct.index)), pivot_pct.index)
    plt.colorbar(label="% within persona")
    plt.title("Dormancy concentration by persona lane")
    plt.tight_layout()
    plt.show()
    display(lane_dorm.sort_values(["persona", "developers"], ascending=[True, False]))
else:
    print("Could not find persona and dormancy_status in the final profile table.")


**What this section looks at:** It shows whether certain lanes over-index in Active, At-Risk, Dormant, or Unactivated populations.

**Exact features / columns used:** `persona` and `dormancy_status` from the final profile table, usually `dev_profile_final_v3`.

**Why it matters:** A heatmap makes risk concentration much easier to spot than raw tables. This is useful for both product insight and prioritization of intervention strategies.

---
## Section 18 — Transition Pattern Overview

This section summarizes the most common movement patterns across the cumulative journey windows.

In [ ]:
if "dev_transition_v1" in available:
    trans_cols = set(table_columns("dev_transition_v1"))
    if {"state_30d", "state_90d"}.issubset(trans_cols):
        trans = con.execute("""
            SELECT state_90d AS from_state, state_30d AS to_state, COUNT(*) AS developers
            FROM dev_transition_v1
            GROUP BY state_90d, state_30d
        """).df()
        display(trans.sort_values("developers", ascending=False).head(25))
        pivot = trans.pivot_table(index="from_state", columns="to_state", values="developers", fill_value=0)
        pivot_pct = pivot.div(pivot.sum(axis=1), axis=0) * 100
        plt.figure(figsize=(9, 7))
        plt.imshow(pivot_pct.fillna(0), aspect="auto", cmap="Blues")
        plt.xticks(range(len(pivot_pct.columns)), pivot_pct.columns, rotation=25, ha="right")
        plt.yticks(range(len(pivot_pct.index)), pivot_pct.index)
        plt.colorbar(label="Row %")
        plt.title("90d to 30d journey transition heatmap")
        plt.tight_layout()
        plt.show()
    elif "transition_90d_to_30d" in trans_cols:
        trans = con.execute("""
            SELECT transition_90d_to_30d, COUNT(*) AS developers
            FROM dev_transition_v1
            GROUP BY transition_90d_to_30d
            ORDER BY developers DESC
        """).df()
        display(trans)
        plt.figure(figsize=(8, 4))
        plt.bar(trans["transition_90d_to_30d"], trans["developers"], color="#42A5F5")
        plt.xticks(rotation=20, ha="right")
        plt.ylabel("Developers")
        plt.title("90d to 30d transition type distribution")
        plt.tight_layout()
        plt.show()
    else:
        print("Could not find compatible transition columns in dev_transition_v1.")
else:
    print("dev_transition_v1 not available.")


**What this section looks at:** It summarizes whether developers are progressing, staying stable, regressing, or moving into dormancy between windows.

**Exact features / columns used:** Preferably `state_90d` and `state_30d` from `dev_transition_v1`; if those are unavailable, it falls back to `transition_90d_to_30d` from the same table.

**Why it matters:** This is the strongest EDA for understanding motion in the system. It tells you where developers tend to get stuck and whether the journey framework captures meaningful progression dynamics.

---
## Section 19 — Summary Statistics by Journey Stage and Lane

This section compares the main engineered feature magnitudes across journey stages and across persona lanes.

In [ ]:
if final_t:
    final_cols = set(table_columns(final_t))
    stage_col = pick_first_column(final_t, ["current_journey_state_30d", "recent_journey_state_90d", "journey_state"])
    metric_cols = [c for c in ["activity_count_30d", "activity_count_90d", "activity_count_180d", "lifetime_activity_count", "lifetime_build_count", "lifetime_champion_count"] if c in final_cols]
    if stage_col and metric_cols:
        stage_expr = ', '.join([f"AVG({c}) AS {c}__avg" for c in metric_cols] + [f"MEDIAN({c}) AS {c}__median" for c in metric_cols])
        stage_stats = con.execute(f"SELECT {stage_col} AS segment, {stage_expr} FROM {final_t} WHERE {stage_col} IS NOT NULL GROUP BY 1 ORDER BY 1").df()
        print("By journey stage")
        display(stage_stats)
    if "persona" in final_cols and metric_cols:
        lane_expr = ', '.join([f"AVG({c}) AS {c}__avg" for c in metric_cols] + [f"MEDIAN({c}) AS {c}__median" for c in metric_cols])
        lane_stats = con.execute(f"SELECT persona AS segment, {lane_expr} FROM {final_t} WHERE persona IS NOT NULL GROUP BY 1 ORDER BY 1").df()
        print("By persona lane")
        display(lane_stats)
else:
    print("Final profile table not available.")


**What this section looks at:** It gives average and median feature magnitudes broken out by journey stage and persona lane.

**Exact features / columns used:** Stage split uses one of `current_journey_state_30d`, `recent_journey_state_90d`, or `journey_state` from the final profile table. Numeric features summarized are `activity_count_30d`, `activity_count_90d`, `activity_count_180d`, `lifetime_activity_count`, `lifetime_build_count`, and `lifetime_champion_count` when present.

**Why it matters:** The visuals tell you the pattern, but summary statistics make it easier to report exact differences and compare feature intensity across your behavioral segments.

---
## Section 20 — Activation / Dormancy Contrast on Core Features

This section contrasts the most important core features across the activation or dormancy groups.

In [ ]:
if final_t:
    final_cols = set(table_columns(final_t))
    segment_col = pick_first_column(final_t, ["dormancy_status", "is_activated"])
    metric_cols = [c for c in ["activity_count_30d", "activity_count_90d", "activity_count_180d", "lifetime_activity_count", "lifetime_build_count", "lifetime_champion_count"] if c in final_cols]
    if segment_col and metric_cols:
        expr = ', '.join([f"AVG({c}) AS {c}" for c in metric_cols])
        seg = con.execute(f"SELECT {segment_col} AS segment, {expr} FROM {final_t} WHERE {segment_col} IS NOT NULL GROUP BY 1 ORDER BY 1").df()
        plot_df = seg.set_index("segment")
        norm = plot_df.div(plot_df.max(axis=0).replace(0, np.nan), axis=1).fillna(0)
        norm.T.plot(kind="bar", figsize=(12, 5), edgecolor="white")
        plt.ylabel("Relative average (feature-normalized)")
        plt.title("Core feature contrast by activation / dormancy segment")
        plt.xticks(rotation=30, ha="right")
        plt.legend(title="Segment", bbox_to_anchor=(1.02, 1), loc="upper left")
        plt.tight_layout()
        plt.show()
        display(seg)
    else:
        print("No compatible activation/dormancy segment column or feature metrics found.")
else:
    print("Final profile table not available.")


**What this section looks at:** It compares the core activity and maturity features across the activation or dormancy groups.

**Exact features / columns used:** Segmentation uses `dormancy_status` or, if needed, `is_activated` from the final profile table. Compared feature columns are `activity_count_30d`, `activity_count_90d`, `activity_count_180d`, `lifetime_activity_count`, `lifetime_build_count`, and `lifetime_champion_count` when available.

**Why it matters:** This shows whether the engineered features really distinguish healthy, activated developers from dormant or at-risk populations, which is critical for the usefulness of the feature framework.

In [ ]:
con.close()
print("Done.")